# XGBoost Training - Occupancy Prediction
Trains an xgboost classifier to predict bus occupancy level (low, medium, high, very_high).

In [1]:
# --- setup: mount google drive and sync repo ---
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_PATH = "/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone"
%cd {REPO_PATH}

!git config --global user.email "eleonorvilla2003@gmail.com"
!git config --global user.name "victoriaeleonor"

!git pull

Mounted at /content/drive
/content/drive/MyDrive/Occupancy_capstone/occupancy-prediction-capstone
Already up to date.


In [2]:
# --- imports ---
import pandas as pd
import numpy as np
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight
from xgboost import XGBClassifier

print("Libraries imported successfully")

Libraries imported successfully


In [3]:
# --- configuration ---

# paths to X and y files generated by the feature engineering notebook
DATASET_PATH = "/content/drive/MyDrive/Occupancy_capstone/Dataset"

# where to save the trained model and results
MODEL_PATH = "/content/drive/MyDrive/Occupancy_capstone/Model"

USE_LAGS = True
suffix   = "with_lags" if USE_LAGS else "no_lags"

X_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_X.parquet"
Y_PATH = f"{DATASET_PATH}/sunt_2024_03_march_{suffix}_y.pkl"

# train/test split
TEST_SIZE    = 0.2
RANDOM_STATE = 42

# xgboost hyperparameters
N_ESTIMATORS         = 300
MAX_DEPTH            = 8
LEARNING_RATE        = 0.05
SUBSAMPLE            = 0.8
COLSAMPLE            = 0.8
MIN_CHILD_WEIGHT     = 5

# stop training if validation loss does not improve after this many rounds
EARLY_STOPPING_ROUNDS = 20

print(f"X path : {X_PATH}")
print(f"y path : {Y_PATH}")
print(f"model  : {MODEL_PATH}")

X path : /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_X.parquet
y path : /content/drive/MyDrive/Occupancy_capstone/Dataset/sunt_2024_03_march_y.pkl
model  : /content/drive/MyDrive/Occupancy_capstone/Model


In [4]:
# --- load data ---

X = pd.read_parquet(X_PATH)
with open(Y_PATH, 'rb') as f:
    y = pickle.load(f)

print("Dataset loaded:")
print(f"  X shape  : {X.shape}")
print(f"  y shape  : {y.shape}")
print(f"  features : {list(X.columns)}")
print(f"\nTarget distribution:")
dist = y.value_counts(normalize=True).sort_index()
for cls, pct in dist.items():
    count = (y == cls).sum()
    bar = '█' * int(pct * 50)
    print(f"  {cls:12s}: {count:>10,} ({pct*100:5.2f}%) {bar}")

Dataset loaded:
  X shape  : (3022798, 11)
  y shape  : (3022798,)
  features : ['route_short_name', 'direction_id', 'pt_sequence', 'stop_id', 'hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'route_progression', 'loading_lag_1', 'loading_lag_2']

Target distribution:
  high        :    552,506 (18.28%) █████████
  low         :  1,341,277 (44.37%) ██████████████████████
  medium      :    892,718 (29.53%) ██████████████
  very_high   :    236,297 ( 7.82%) ███


In [5]:
# --- analyze class imbalance and compute class weights ---
# minority classes get higher weights so xgboost pays more attention to them
# this balances the model without discarding any data

class_counts = y.value_counts()
imbalance_ratio = class_counts.max() / class_counts.min()

print(f"Imbalance ratio : {imbalance_ratio:.1f}:1")
print(f"Majority class  : {class_counts.idxmax()} ({class_counts.max():,})")
print(f"Minority class  : {class_counts.idxmin()} ({class_counts.min():,})")

classes = np.unique(y)
class_weights_array = compute_class_weight('balanced', classes=classes, y=y)
class_weights = dict(zip(classes, class_weights_array))

print(f"\nClass weights:")
for cls, weight in sorted(class_weights.items()):
    print(f"  {cls:12s}: {weight:.4f}")

Imbalance ratio : 5.7:1
Majority class  : low (1,341,277)
Minority class  : very_high (236,297)

Class weights:
  high        : 1.3678
  low         : 0.5634
  medium      : 0.8465
  very_high   : 3.1981


In [6]:
# --- encode target and split data ---

# xgboost requires numeric labels (0, 1, 2, 3)
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

print("Label encoding:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {cls:12s} -> {i}")

# stratified split preserves class distribution in both sets
X_train, X_test, y_train_enc, y_test_enc = train_test_split(
    X, y_encoded,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_encoded
)

# keep original string labels for reporting
_, _, y_train_orig, y_test_orig = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"\nTrain set : {len(X_train):,} records")
print(f"Test set  : {len(X_test):,} records")

Label encoding:
  high         -> 0
  low          -> 1
  medium       -> 2
  very_high    -> 3

Train set : 2,418,238 records
Test set  : 604,560 records


In [7]:
# --- compute sample weights for training set ---
# one weight per sample, passed to model.fit() to adjust the loss function

sample_weights_train = compute_sample_weight(class_weights, y_train_orig)

print(f"Sample weights computed:")
print(f"  total samples : {len(sample_weights_train):,}")
print(f"  min weight    : {sample_weights_train.min():.4f}")
print(f"  max weight    : {sample_weights_train.max():.4f}")
print(f"  weight ratio  : {sample_weights_train.max() / sample_weights_train.min():.2f}:1")

Sample weights computed:
  total samples : 2,418,238
  min weight    : 0.5634
  max weight    : 3.1981
  weight ratio  : 5.68:1


In [8]:
# --- train xgboost model ---

model = XGBClassifier(
    objective='multi:softprob',
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    subsample=SUBSAMPLE,
    colsample_bytree=COLSAMPLE,
    min_child_weight=MIN_CHILD_WEIGHT,
    gamma=0.1,
    reg_lambda=1,
    tree_method='hist',
    eval_metric='mlogloss',
    early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=1
)

print("XGBoost configuration:")
print(f"  n_estimators (max)    : {N_ESTIMATORS}")
print(f"  max_depth             : {MAX_DEPTH}")
print(f"  learning_rate         : {LEARNING_RATE}")
print(f"  early_stopping_rounds : {EARLY_STOPPING_ROUNDS}")
print(f"  sample_weight         : enabled")
print("\nTraining...")

start_time = time.time()

model.fit(
    X_train,
    y_train_enc,
    sample_weight=sample_weights_train,
    # monitor test loss to trigger early stopping
    eval_set=[(X_test, y_test_enc)],
    verbose=50
)

elapsed = time.time() - start_time
print(f"\nTraining completed in {elapsed:.2f}s ({elapsed/60:.1f} min)")
print(f"Best iteration: {model.best_iteration} (out of {N_ESTIMATORS} max)")

XGBoost configuration:
  n_estimators (max)    : 300
  max_depth             : 8
  learning_rate         : 0.05
  early_stopping_rounds : 20
  sample_weight         : enabled

Training...
[0]	validation_0-mlogloss:1.30771
[50]	validation_0-mlogloss:0.29100
[100]	validation_0-mlogloss:0.20900
[150]	validation_0-mlogloss:0.19457
[200]	validation_0-mlogloss:0.18878
[250]	validation_0-mlogloss:0.18530
[299]	validation_0-mlogloss:0.18186

Training completed in 489.14s (8.2 min)
Best iteration: 299 (out of 300 max)


In [9]:
# --- evaluate model ---
# primary metric: f1 macro (treats all classes equally regardless of size)
# secondary metric: balanced accuracy (average recall per class)

y_pred_train = label_encoder.inverse_transform(model.predict(X_train))
y_pred_test  = label_encoder.inverse_transform(model.predict(X_test))

metrics = {
    'accuracy'         : (accuracy_score(y_train_orig, y_pred_train),          accuracy_score(y_test_orig, y_pred_test)),
    'balanced accuracy': (balanced_accuracy_score(y_train_orig, y_pred_train),  balanced_accuracy_score(y_test_orig, y_pred_test)),
    'f1 macro'         : (f1_score(y_train_orig, y_pred_train, average='macro'), f1_score(y_test_orig, y_pred_test, average='macro')),
}

print(f"{'Metric':<22} {'Train':>10} {'Test':>10} {'Gap':>10}")
print('-' * 55)
for name, (train_val, test_val) in metrics.items():
    print(f"{name:<22} {train_val:>10.4f} {test_val:>10.4f} {train_val - test_val:>10.4f}")

gap = metrics['balanced accuracy'][0] - metrics['balanced accuracy'][1]
if gap > 0.1:
    print(f"\nWarning: high gap ({gap:.2%}) — possible overfitting")
elif gap > 0.05:
    print(f"\nModerate gap ({gap:.2%}) — acceptable")
else:
    print(f"\nLow gap ({gap:.2%}) — model generalizes well")

Metric                      Train       Test        Gap
-------------------------------------------------------
accuracy                   0.9346     0.9338     0.0008
balanced accuracy          0.9312     0.9299     0.0013
f1 macro                   0.9273     0.9262     0.0011

Low gap (0.13%) — model generalizes well


In [10]:
# --- classification report ---
print("Classification Report (test set):")
print(classification_report(y_test_orig, y_pred_test))

Classification Report (test set):
              precision    recall  f1-score   support

        high       0.91      0.90      0.91    110501
         low       0.96      0.96      0.96    268256
      medium       0.91      0.90      0.91    178544
   very_high       0.91      0.95      0.93     47259

    accuracy                           0.93    604560
   macro avg       0.92      0.93      0.93    604560
weighted avg       0.93      0.93      0.93    604560



In [11]:
# --- confusion matrix ---

labels = sorted(y.unique())
cm = confusion_matrix(y_test_orig, y_pred_test, labels=labels)

print("Confusion Matrix (test set):")
print(f"\n{'':>12}", end='')
for label in labels:
    print(f"{label:>12}", end='')
print("  <- predicted")
print('-' * (12 + 12 * len(labels)))
for i, label in enumerate(labels):
    print(f"{label:>12}", end='')
    for j in range(len(labels)):
        print(f"{cm[i,j]:>12,}", end='')
    print("  | actual")

print("\nPer-class accuracy:")
for i, label in enumerate(labels):
    total   = cm[i, :].sum()
    correct = cm[i, i]
    acc     = correct / total if total > 0 else 0
    status  = 'OK' if acc > 0.6 else 'LOW' if acc > 0.4 else 'POOR'
    print(f"  {status:4s} {label:12s}: {acc:.2%} ({correct:,}/{total:,})")

Confusion Matrix (test set):

                    high         low      medium   very_high  <- predicted
------------------------------------------------------------
        high      99,820          75       6,109       4,497  | actual
         low          56     258,579       9,620           1  | actual
      medium       7,763       9,436     161,319          26  | actual
   very_high       2,387           5          33      44,834  | actual

Per-class accuracy:
  OK   high        : 90.33% (99,820/110,501)
  OK   low         : 96.39% (258,579/268,256)
  OK   medium      : 90.35% (161,319/178,544)
  OK   very_high   : 94.87% (44,834/47,259)


In [12]:
# --- feature importance ---

importances   = model.feature_importances_
feature_names = X.columns.tolist()
indices       = np.argsort(importances)[::-1]

print("Top features by importance:")
print('-' * 55)
for i in range(min(15, len(feature_names))):
    idx = indices[i]
    bar = '█' * int(importances[idx] * 50)
    print(f"{i+1:2d}. {feature_names[idx]:<25} {importances[idx]:.4f}  {bar}")

cumsum = 0
for i, idx in enumerate(indices):
    cumsum += importances[idx]
    if cumsum >= 0.8:
        print(f"\nTop {i+1} features explain 80% of total importance")
        break

Top features by importance:
-------------------------------------------------------
 1. loading_lag_1             0.7520  █████████████████████████████████████
 2. loading_lag_2             0.1494  ███████
 3. hour                      0.0207  █
 4. route_progression         0.0192  
 5. direction_id              0.0116  
 6. pt_sequence               0.0103  
 7. is_weekend                0.0092  
 8. route_short_name          0.0082  
 9. is_rush_hour              0.0069  
10. stop_id                   0.0066  
11. day_of_week               0.0058  

Top 2 features explain 80% of total importance


In [17]:
# --- compare with baseline and random forest ---
# baseline: always predict the majority class (simplest possible model)
# rf results are loaded from file if available (saved by the rf training notebook)
# otherwise falls back to hardcoded values from the original run

majority_class = y_train_orig.value_counts().idxmax()
baseline_acc   = (y_test_orig == majority_class).mean()

RF_RESULTS_PATH = f"{MODEL_PATH}/rf_results_{suffix}.pkl"
if os.path.exists(RF_RESULTS_PATH):
    with open(RF_RESULTS_PATH, 'rb') as f:
        rf_results  = pickle.load(f)
    rf_acc      = rf_results['accuracy']
    rf_bal_acc  = rf_results['balanced_accuracy']
    rf_f1_macro = rf_results['f1_macro']
    print("RF results loaded from file")
else:
    rf_acc      = 0.5920
    rf_bal_acc  = 0.5677
    rf_f1_macro = 0.4753
    print("RF results using hardcoded values (run TrainingRandomForest first to update)")

xgb_acc      = metrics['accuracy'][1]
xgb_bal_acc  = metrics['balanced accuracy'][1]
xgb_f1_macro = metrics['f1 macro'][1]

print(f"\n{'Model':<20} {'Accuracy':>10} {'Bal. Acc':>10} {'F1 Macro':>10}")
print('-' * 55)
print(f"{'Baseline':<20} {baseline_acc:>10.4f} {'—':>10} {'—':>10}")
print(f"{'Random Forest':<20} {rf_acc:>10.4f} {rf_bal_acc:>10.4f} {rf_f1_macro:>10.4f}")
print(f"{'XGBoost':<20} {xgb_acc:>10.4f} {xgb_bal_acc:>10.4f} {xgb_f1_macro:>10.4f}")
print(f"\nXGBoost vs Random Forest:")
print(f"  balanced accuracy : {(xgb_bal_acc - rf_bal_acc)*100:+.2f}%")
print(f"  f1 macro          : {(xgb_f1_macro - rf_f1_macro)*100:+.2f}%")

RF results loaded from file

Model                  Accuracy   Bal. Acc   F1 Macro
-------------------------------------------------------
Baseline                 0.4437          —          —
Random Forest            0.9331     0.9273     0.9252
XGBoost                  0.9338     0.9299     0.9262

XGBoost vs Random Forest:
  balanced accuracy : +0.25%
  f1 macro          : +0.10%


In [18]:
# --- save model and results to drive ---
model_file = f"{MODEL_PATH}/xgboost_occupancy_{suffix}.pkl"
with open(model_file, 'wb') as f:
    pickle.dump(model, f)
print(f"Model saved        -> {model_file}")

encoder_file = f"{MODEL_PATH}/xgboost_label_encoder_{suffix}.pkl"
with open(encoder_file, 'wb') as f:
    pickle.dump(label_encoder, f)
print(f"Label encoder      -> {encoder_file}")

features_file = f"{MODEL_PATH}/xgboost_feature_names_{suffix}.pkl"
with open(features_file, 'wb') as f:
    pickle.dump(feature_names, f)
print(f"Feature names      -> {features_file}")

xgb_results = {
    'accuracy'         : xgb_acc,
    'balanced_accuracy': xgb_bal_acc,
    'f1_macro'         : xgb_f1_macro,
}
xgb_results_file = f"{MODEL_PATH}/xgb_results_{suffix}.pkl"
with open(xgb_results_file, 'wb') as f:
    pickle.dump(xgb_results, f)
print(f"XGBoost results    -> {xgb_results_file}")

Model saved        -> /content/drive/MyDrive/Occupancy_capstone/Model/xgboost_occupancy.pkl
Label encoder      -> /content/drive/MyDrive/Occupancy_capstone/Model/xgboost_label_encoder.pkl
Feature names      -> /content/drive/MyDrive/Occupancy_capstone/Model/xgboost_feature_names.pkl
XGBoost results    -> /content/drive/MyDrive/Occupancy_capstone/Model/xgb_results.pkl


In [19]:
# --- push changes to github ---
!git add .
!git commit -m "update: xgboost training with early stopping, drive saving and consistent metrics"
!git push

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
fatal: Unable to read current working directory: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
fatal: Unable to read current working directory: Transport endpoint is not connected
shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
fatal: Unable to read current working directory: Transport endpoint is not connected
